# Detecção de Fraudes em Cartões de Crédito com Machine Learning Explicável

Este notebook apresenta um pipeline completo para detectar fraudes em transações reais de cartão de crédito utilizando **Python**, **pandas**, **NumPy**, **scikit-learn**, **XGBoost** e **SHAP**.

Como apenas **0,173%** das transações são fraudulentas, a acurácia torna-se uma métrica enganosa. Por isso, o projeto prioriza o **Recall** da classe positiva (fraude), a área sob a curva **Precision-Recall (PR-AUC)**, o ajuste estratégico do **limiar de decisão (threshold)** e a explicabilidade das decisões.

## 0. Configuração e Importação das Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Coleta e Análise Exploratória (EDA)

Nesta etapa, carregamos o dataset público diretamente pelo link oficial, verificamos a integridade dos dados e analisamos o desbalanceamento severo entre transações legítimas (`0`) e fraudulentas (`1`).

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
df = pd.read_csv(url)

print(f'Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas')
print(f'Total de valores nulos: {df.isnull().sum().sum()}')

contagem = df['Class'].value_counts()
percentual = df['Class'].value_counts(normalize=True) * 100

print('\nDistribuição das Classes:')
print(f'Legítimas (0): {contagem[0]} ({percentual[0]:.3f}%)')
print(f'Fraudes   (1): {contagem[1]} ({percentual[1]:.3f}%)')

print('\nEstatísticas do Valor (Amount) por Classe:')
print(df.groupby('Class')['Amount'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(x='Class', data=df, ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribuição das Classes (Escala Log)')
axes[0].set_yscale('log')
axes[0].set_ylabel('Contagem')

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1], showfliers=False, palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Distribuição de Valores (Amount) sem Outliers Extremos')
plt.tight_layout()
plt.show()

### Por que descartar a Acurácia?
Como **99,827%** das operações são legítimas, um modelo ingênuo que classifique todas as transações como `0` atingiria **99,83% de acurácia**, mas deixaria passar **100% das fraudes**. Por isso, avaliamos os algoritmos pelo **Recall** e pela curva **Precision-Recall (PR-AUC)**.

## 2. Pré-processamento e Divisão Estratégica

Realizamos a divisão estratificada em treino (80%) e teste (20%) para manter exatamente a mesma proporção de fraudes nos dois conjuntos. Em seguida, aplicamos o `RobustScaler` apenas nas colunas `Time` e `Amount` (as colunas `V1` a `V28` já passaram por transformação PCA), ajustando o escalonador exclusivamente nos dados de treino para evitar *Data Leakage*.

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

print(f'Fraudes no treino: {y_train.sum()} ({y_train.mean()*100:.3f}%)')
print(f'Fraudes no teste:  {y_test.sum()} ({y_test.mean()*100:.3f}%)')

## 3. Treinamento e Comparação de Modelos

Treinamos três abordagens distintas utilizando ponderação nativa para classes desbalanceadas (`class_weight='balanced'` e `scale_pos_weight`), evitando a geração artificial de dados sintéticos antes da validação:
- **Regressão Logística** (*baseline* linear)
- **Random Forest** (*bagging* de árvores de decisão)
- **XGBoost** (*gradient boosting* otimizado)

In [ ]:
ratio_desbalanceamento = (y_train == 0).sum() / (y_train == 1).sum()

modelos = {
    'Regressão Logística': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        scale_pos_weight=ratio_desbalanceamento,
        eval_metric='aucpr',
        n_jobs=-1,
        random_state=42
    )
}

resultados = []
probabilidades = {}

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    probabilidades[nome] = y_prob
    
    resultados.append({
        'Modelo': nome,
        'Recall (Fraude)': recall_score(y_test, y_pred),
        'Precision (Fraude)': precision_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'PR-AUC': average_precision_score(y_test, y_prob)
    })

df_resultados = pd.DataFrame(resultados).sort_values(by='PR-AUC', ascending=False)
print(df_resultados.to_string(index=False))

## 4. Ajuste do Limiar de Decisão (Threshold Tuning)

O limiar padrão de `0.50` nem sempre reflete o custo financeiro real de uma operação bancária. Nesta etapa, analisamos o comportamento de **Precision** e **Recall** ao longo de diferentes limiares de probabilidade no **XGBoost** e comparamos o impacto operacional antes e depois do ajuste.

In [ ]:
y_prob_xgb = probabilidades['XGBoost']
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_xgb)

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precisions[:-1], label='Precision (Evitar bloqueio indevido)', color='#2980b9', lw=2)
plt.plot(thresholds, recalls[:-1], label='Recall (Capturar fraudes reais)', color='#c0392b', lw=2)
plt.axvline(0.50, color='gray', linestyle='--', label='Limiar Padrão (0.50)')
plt.axvline(0.85, color='green', linestyle=':', label='Limiar Ajustado (0.85)')
plt.title('Trade-off Precision vs. Recall por Limiar de Decisão (XGBoost)')
plt.xlabel('Limiar de Probabilidade (Threshold)')
plt.ylabel('Score')
plt.legend()
plt.show()

for limiar in [0.50, 0.85]:
    y_pred_ajustado = (y_prob_xgb >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_ajustado).ravel()
    
    print(f'\n--- Threshold = {limiar:.2f} ---')
    print(f'Recall:    {recall_score(y_test, y_pred_ajustado):.2%} ({tp} de {tp+fn} fraudes capturadas)')
    print(f'Precision: {precision_score(y_test, y_pred_ajustado):.2%} ({fp} clientes legítimos bloqueados)')
    print(f'F1-Score:  {f1_score(y_test, y_pred_ajustado):.2%}')

### Justificativa de Negócio para o Limiar
Devido ao uso de `scale_pos_weight`, o XGBoost atribui probabilidades elevadas a casos suspeitos. Ajustar o limiar de `0.50` para `0.85` mantém uma taxa alta de captura de fraudes (**Recall** acima de 80%) enquanto reduz drasticamente os falsos positivos, evitando o bloqueio desnecessário de cartões de clientes legítimos.

## 5. Explicabilidade do Modelo com SHAP

Para atender aos requisitos de auditoria, transparência e confiança da equipe de risco, utilizamos o `TreeExplainer` da biblioteca **SHAP** para interpretar tanto o comportamento global do **XGBoost** quanto os fatores que motivaram o bloqueio de uma fraude específica.

In [ ]:
modelo_xgb = modelos['XGBoost']
explainer = shap.TreeExplainer(modelo_xgb)
shap_values = explainer(X_test)

plt.title('Impacto Global das Variáveis na Probabilidade de Fraude (SHAP)')
shap.summary_plot(shap_values, X_test, max_display=10)

indices_fraude = np.where((y_test == 1) & (y_prob_xgb >= 0.85))[0]
exemplo_idx = indices_fraude[0]

print(f'Explicação Local para a transação fraudulenta de índice {exemplo_idx}:')
shap.plots.waterfall(shap_values[exemplo_idx], max_display=10)